# Holdout-eval sentence ablation — CT Temporal Progression

**Question:** If CE is trained on one set of class sentences, does performance hold when **eval** picks the nearest of 3 classes using **different sentences never used in CE training**?

## Eval paradigm (this notebook only)

- For each finding × class store **5 held-out paraphrase sentences** (disjoint from train CE templates).
- At **every eval example**, randomly sample **one sentence per class**:
  `y_hat = argmax_c cos(d_f, emb(s_c))`
- **Train CE** still uses fixed `PROTO_TRAIN[f]` from a separate train template bank.
- Mag + optional cross-modal SupCon match the proposed pipeline.
- Ablation: **CE × Mag × SupCon** under this eval paradigm only.

| Text role | Source |
|-----------|--------|
| Train CE | `TRAIN_TEMPLATES` → frozen `PROTO_TRAIN[f]` (3×512) |
| Eval readout | `EVAL_BANK[f][c]` = 5 strings; sample 1/class per example |
| SupCon (train) | per-example evidence/dynamic (**not** used at test) |

Protocol: Hub `train_*` train+tune; Hub `valid_*` one-shot test; CT-CLIP frozen.

## 1. Setup: clone CT-CLIP + repo

In [ ]:
%cd /content
![ -d CT-CLIP ] || git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
%cd /content/CT-CLIP
!pip install -q -e transformer_maskgit
!pip install -q -e CT_CLIP
!pip install -q scikit-learn pandas matplotlib tqdm
import sys
for p in ['/content/CT-CLIP/CT_CLIP', '/content/CT-CLIP/transformer_maskgit']:
    if p not in sys.path:
        sys.path.insert(0, p)
%cd /content
![ -d 3dCT ] || git clone https://github.com/nprakash1/3dCT.git
%cd /content/3dCT
!git pull --ff-only || true
import ct_clip, transformer_maskgit
print('CT-CLIP import OK')







## 2. Drive + config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, csv, math, random, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
from sklearn.metrics import f1_score
from itertools import product
import pandas as pd
csv.field_size_limit(10**9)

DRIVE = '/content/drive/MyDrive/3dCT'
IMG_DIR = f'{DRIVE}/ctclip_cache/img'
WEIGHTS = f'{DRIVE}/ctclip_weights'
LAB = '/content/3dCT/medgemma_labels_v3.jsonl'
LAB_DS = '/content/3dCT/medgemma_labels (2).jsonl'
EVAL_BANK_PT = f'{DRIVE}/ctclip_cache/eval_holdout_sentence_bank.pt'

TUNE_FRAC, SPLIT_SEED = 0.15, 2026
REQUIRE_COMPLETE_HUB_VALID_FEATURES = True
CLASSES = ['worsened', 'stable', 'improved']
C2I = {c: i for i, c in enumerate(CLASSES)}
I2C = {i: c for c, i in C2I.items()}

FINDING_CONDITIONING = True
FINDING_AS_4TH_TOKEN = False
USE_LEARNED_FINDING_EMB = True
ANTISYM = False

USE_CE, USE_MAGNITUDE, USE_SUPCON = True, True, False
CONTRASTIVE_SYMMETRIC = False
LEARNABLE_TAU_CON = True
TAU_CON_INIT = 0.07
STABLE_TEXT_SEED = 2026
EVAL_SAMPLE_SEED = 2027
N_EVAL_SENTENCES = 5

D_MODEL, EPOCHS, LR, PATIENCE = 256, 120, 1e-3, 20
WEIGHT_DECAY = 1e-2
LAMBDA_CE, LAMBDA_MAG, LAMBDA_CON = 1.0, 0.5, 0.5
K_FINDINGS_PER_BATCH, N_PER_CLASS, MAX_BATCH_SIZE = 8, 4, 256

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEVICE, 'img', os.path.isdir(IMG_DIR))
print('EVAL: holdout', N_EVAL_SENTENCES, 'sents/class; random sample at eval')

## 3. Image cache, text tower, labels, Hub splits

In [ ]:
import glob
import sys
sys.path.insert(0, '/content/3dCT/scripts')
from huggingface_hub import hf_hub_download
from ctclip_utils import CTCLIPEmbedder, REPO_ID, CTCLIP_WEIGHTS_HF

POOLED = {}
for fp in glob.glob(f'{IMG_DIR}/*.pt'):
    key = os.path.basename(fp).replace('.pt', '')
    POOLED[key] = torch.load(fp, map_location='cpu').float().view(-1)
print('pooled volumes', len(POOLED))
assert len(POOLED) > 100

wp = f'{WEIGHTS}/{os.path.basename(CTCLIP_WEIGHTS_HF)}'
if not os.path.exists(wp):
    wp = hf_hub_download(REPO_ID, CTCLIP_WEIGHTS_HF, repo_type='dataset', local_dir=WEIGHTS)
emb = CTCLIPEmbedder(wp)
print('text tower on', emb.device)

def vkey(v):
    return v.replace('.nii.gz', '').replace('.nii', '')

def volume_domain(v):
    k = vkey(v).lower()
    if k.startswith('train_'): return 'hub_train'
    if k.startswith('valid_'): return 'hub_valid'
    return 'unknown'

def pair_domain(pv, cv):
    a, b = volume_domain(pv), volume_domain(cv)
    return a if a == b else 'cross_domain'

def dev_partition(patient):
    raw = f'{SPLIT_SEED}|{patient}'.encode()
    u = int.from_bytes(hashlib.sha256(raw).digest()[:8], 'big') / 2**64
    return 'tune' if u < TUNE_FRAC else 'train'

recs = {}
for line in open(LAB, encoding='utf-8'):
    if not line.strip(): continue
    x = json.loads(line)
    recs[(x['patient'], x['prior_volume'], x['curr_volume'])] = x

dyn_of = {}
try:
    for line in open(LAB_DS, encoding='utf-8'):
        if not line.strip(): continue
        x = json.loads(line)
        ds = x.get('dynamic_sentences') or []
        if isinstance(ds, list):
            ds = ' '.join(s for s in ds if isinstance(s, str))
        dyn_of[(x['patient'], x['prior_volume'], x['curr_volume'])] = (ds or '').strip()
    print('dynamic for', len(dyn_of), 'pairs')
except FileNotFoundError:
    print('WARN no dynamic file')

pair_ids = {}
examples = {'train': [], 'tune': [], 'test': []}
skipped = Counter()
missing_hub_valid = []
for key, rec in recs.items():
    patient, pv, cv = key
    domain = pair_domain(pv, cv)
    if domain == 'hub_train':
        sp = dev_partition(patient)
    elif domain == 'hub_valid':
        sp = 'test'
    else:
        skipped[domain] += 1
        continue
    if not rec.get('parse_ok'):
        skipped['no_label'] += 1
        continue
    if vkey(pv) not in POOLED or vkey(cv) not in POOLED:
        skipped['no_feat'] += 1
        if sp == 'test':
            missing_hub_valid.append(key)
        continue
    for fd in rec.get('findings', []):
        if fd.get('tier') != 'explicit':
            continue
        d, f = fd.get('direction'), fd.get('finding')
        if d not in C2I or not f:
            continue
        pid = pair_ids.setdefault(key, len(pair_ids))
        examples[sp].append(dict(
            vp=vkey(pv), vc=vkey(cv), patient=patient, finding=f,
            hub_domain=domain, y=C2I[d], pid=pid,
            evidence=fd.get('evidence', '') or '',
            dynamic=dyn_of.get(key, ''),
        ))

assert examples['train'] and examples['tune'] and examples['test']
if REQUIRE_COMPLETE_HUB_VALID_FEATURES:
    assert not missing_hub_valid

FINDINGS = [
    'Medical material', 'Arterial wall calcification', 'Cardiomegaly',
    'Pericardial effusion', 'Coronary artery wall calcification', 'Hiatal hernia',
    'Lymphadenopathy', 'Emphysema', 'Atelectasis', 'Lung nodule', 'Lung opacity',
    'Pulmonary fibrotic sequela', 'Pleural effusion', 'Mosaic attenuation pattern',
    'Peribronchial thickening', 'Consolidation', 'Bronchiectasis',
    'Interlobular septal thickening',
]
F2I = {f: i for i, f in enumerate(FINDINGS)}
for sp in examples:
    for e in examples[sp]:
        e['fid'] = F2I[e['finding']]
for sp in ['train', 'tune', 'test']:
    cc = Counter(e['y'] for e in examples[sp])
    print(f"{sp:5}: {len(examples[sp]):5}  w/s/i={cc[0]}/{cc[1]}/{cc[2]}")
print('skipped', dict(skipped))

## 4. Train CE prototypes vs holdout eval sentence bank

**Train CE:** fixed templates → `PROTO_TRAIN[f]` (3 vectors).  
**Eval:** 5 held-out paraphrases per finding×class (strings never equal to train templates).  
At eval we sample one embedding per class (cached).

In [ ]:
# ---- TRAIN templates (CE only; must not overlap eval bank strings) ----
TRAIN_TEMPLATES = {
    'worsened': [
        '{f} has increased compared to the prior study',
        '{f} has worsened since the previous exam',
        'interval enlargement of {f}',
        'new {f}',
        'increased {f}',
    ],
    'stable': [
        '{f} is unchanged compared to the prior study',
        'stable {f} with no interval change',
        'no significant change in {f}',
        '{f} appears similar to prior',
    ],
    'improved': [
        '{f} has decreased compared to the prior study',
        '{f} has improved since the previous exam',
        'interval decrease of {f}',
        '{f} has resolved',
        'decreased {f}',
    ],
}

# Holdout eval bank: 5 MedGemma-style paraphrases/class, disjoint from TRAIN_TEMPLATES
EVAL_SENTENCE_STEMS = {
    'worsened': [
        'There is interval progression of the {f}.',
        'The {f} is more conspicuous than on the prior examination.',
        'Findings suggest worsening of the {f} relative to prior.',
        'Compared with the previous study, the {f} has enlarged.',
        'Progressive {f} is noted since the last CT.',
    ],
    'stable': [
        'No appreciable interval change in the {f}.',
        'The {f} is similar in appearance to the prior exam.',
        'Chronic-appearing {f} without definite interval change.',
        'The {f} remains essentially unchanged from prior.',
        'Stability of the {f} is demonstrated compared with prior imaging.',
    ],
    'improved': [
        'There is interval improvement of the {f}.',
        'The {f} has decreased in extent since the prior study.',
        'Partial resolution of the {f} compared with prior.',
        'Findings indicate improving {f} relative to the previous CT.',
        'The {f} is less pronounced than on the prior examination.',
    ],
}

STABLE_SYNTH_TEMPLATES = [
    'The {f} is unchanged from the prior examination.',
    'There has been no significant interval change in the {f}.',
    'The {f} remains stable compared with the prior examination.',
    'The {f} is stable compared with the previous study.',
]

def l2np(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

def embed_mean(texts):
    texts = [t for t in texts if t and str(t).strip()]
    if not texts:
        return None
    vecs = []
    for i in range(0, len(texts), 64):
        vecs.append(emb.embed_texts(texts[i:i+64], normalize=True).numpy())
    return l2np(np.concatenate(vecs, 0).mean(0))

def embed_list(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        out.append(emb.embed_texts(texts[i:i+bs], normalize=True).float().cpu())
    return torch.cat(out, 0) if out else torch.zeros(0, 512)

PROTO_TRAIN = {}
for f in FINDINGS:
    rows = []
    for c in CLASSES:
        prompts = [t.format(f=f.lower()) for t in TRAIN_TEMPLATES[c]]
        rows.append(embed_mean(prompts))
    PROTO_TRAIN[f] = torch.tensor(np.stack(rows)).float()
print('PROTO_TRAIN', len(PROTO_TRAIN), PROTO_TRAIN[FINDINGS[0]].shape)

train_string_set = set()
for c, ts_ in TRAIN_TEMPLATES.items():
    for t in ts_:
        for f in FINDINGS:
            train_string_set.add(t.format(f=f.lower()).strip().lower())

EVAL_BANK = {}
EVAL_BANK_EMB = {}
overlap = 0
for f in FINDINGS:
    per_c_txt, per_c_emb = [], []
    for c in CLASSES:
        stems = EVAL_SENTENCE_STEMS[c]
        assert len(stems) >= N_EVAL_SENTENCES
        sents = [stems[i].format(f=f.lower()) for i in range(N_EVAL_SENTENCES)]
        for s in sents:
            if s.strip().lower() in train_string_set:
                overlap += 1
        per_c_txt.append(sents)
        per_c_emb.append(embed_list(sents))
    EVAL_BANK[f] = per_c_txt
    EVAL_BANK_EMB[f] = torch.stack(per_c_emb, 0)
assert overlap == 0, 'train/eval sentence overlap=%d' % overlap
torch.save(dict(bank=EVAL_BANK, emb=EVAL_BANK_EMB), EVAL_BANK_PT)
print('EVAL_BANK ok; overlap=', overlap, 'emb', EVAL_BANK_EMB[FINDINGS[0]].shape)

# Temporal text for SupCon
rng = random.Random(STABLE_TEXT_SEED)

def _real(e):
    return (e.get("evidence") or "").strip() or (e.get("dynamic") or "").strip()

for sp, exs in examples.items():
    for e in exs:
        real = _real(e)
        if real:
            e["temporal_text"] = real
            e["temporal_src"] = "real"
        elif e["y"] == C2I["stable"]:
            tid = rng.randrange(len(STABLE_SYNTH_TEMPLATES))
            e["temporal_text"] = STABLE_SYNTH_TEMPLATES[tid].format(f=e["finding"].lower())
            e["temporal_src"] = "synthetic"
        else:
            c = I2C[e["y"]]
            e["temporal_text"] = TRAIN_TEMPLATES[c][0].format(f=e["finding"].lower())
            e["temporal_src"] = "template_fallback"

all_t, meta = [], []
for sp in ["train", "tune", "test"]:
    for i, e in enumerate(examples[sp]):
        all_t.append(e["temporal_text"])
        meta.append((sp, i))
uniq, uix = [], {}
for t in all_t:
    if t not in uix:
        uix[t] = len(uniq)
        uniq.append(t)
print("encoding", len(uniq), "unique temporal texts...")
U = embed_list(uniq)
by = {sp: [None]*len(examples[sp]) for sp in examples}
for (sp, i), t in zip(meta, all_t):
    by[sp][i] = U[uix[t]]
TEMPORAL_TEXT_EMB = {sp: torch.stack(by[sp], 0) for sp in by}

def tensorize(exs, tt):
    return {
        "vp": torch.stack([POOLED[e["vp"]] for e in exs]),
        "vc": torch.stack([POOLED[e["vc"]] for e in exs]),
        "pr": torch.stack([PROTO_TRAIN[e["finding"]] for e in exs]),
        "y": torch.tensor([e["y"] for e in exs], dtype=torch.long),
        "fid": torch.tensor([e["fid"] for e in exs], dtype=torch.long),
        "tt": tt.float().clone(),
        "fn": [e["finding"] for e in exs],
    }

DATA = {sp: tensorize(examples[sp], TEMPORAL_TEXT_EMB[sp]) for sp in ["train", "tune", "test"]}
cnt = Counter(DATA["train"]["y"].tolist()); tot = sum(cnt.values())
W_cls = torch.tensor([tot / (3 * max(cnt[i], 1)) for i in range(3)], dtype=torch.float32)
print("weights", [round(x,3) for x in W_cls.tolist()])
for sp in DATA:
    print(sp, {k: (tuple(v.shape) if torch.is_tensor(v) else len(v)) for k,v in DATA[sp].items()})

## 5. Model + holdout eval + sampler

Train CE uses `PROTO_TRAIN`.  
Eval samples one of 5 holdout sentence embs per class (never seen in CE training).

In [ ]:
class DifferenceTransformer(nn.Module):
    def __init__(self, n_findings=18, d_in=512, d_model=256, n_layers=2, n_heads=4,
                 dropout=0.1, antisym=False, magnitude=False,
                 finding_conditioning=True, finding_as_4th_token=False,
                 use_learned_finding_emb=True, frozen_finding_emb=None,
                 tau_con_init=0.07, learnable_tau_con=True):
        super().__init__()
        self.finding_conditioning = finding_conditioning
        self.finding_as_4th_token = finding_as_4th_token and finding_conditioning
        self.antisym = antisym
        self.W = nn.Linear(d_in, d_model)
        self.role = nn.Parameter(torch.randn(2, d_model) * 0.02)
        self.e_diff = nn.Parameter(torch.randn(1, d_model) * 0.02)
        if finding_conditioning and use_learned_finding_emb:
            self.finding_emb = nn.Embedding(n_findings, d_model)
            nn.init.normal_(self.finding_emb.weight, std=0.02)
        else:
            self.finding_emb = None
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, dropout=dropout,
            batch_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, d_in)
        self.mag_head = nn.Linear(d_model, 1) if magnitude else None
        self.logit_scale = nn.Parameter(torch.tensor(float(np.log(1 / 0.07))))
        log_tau = float(np.log(tau_con_init))
        if learnable_tau_con:
            self.log_tau_con = nn.Parameter(torch.tensor(log_tau))
        else:
            self.register_buffer('log_tau_con', torch.tensor(log_tau))

    def _pass(self, vp, vc, fid):
        B = vp.size(0)
        tp = self.W(vp) + self.role[0]
        tc = self.W(vc) + self.role[1]
        ed = self.e_diff.expand(B, -1).clone()
        if self.finding_emb is not None and not self.finding_as_4th_token:
            ed = ed + self.finding_emb(fid)
            seq = torch.stack([ed, tp, tc], 1)
        elif self.finding_emb is not None and self.finding_as_4th_token:
            seq = torch.stack([ed, tp, tc, self.finding_emb(fid)], 1)
        else:
            seq = torch.stack([ed, tp, tc], 1)
        h = self.enc(seq)[:, 0]
        mag = self.mag_head(h).squeeze(-1) if self.mag_head is not None else None
        return self.head(h), mag

    def forward(self, vp, vc, fid=None):
        if fid is None:
            fid = torch.zeros(vp.size(0), dtype=torch.long, device=vp.device)
        vd, mag = self._pass(vp, vc, fid)
        if self.antisym:
            vr, _ = self._pass(vc, vp, fid)
            vd = vd - vr
        return vd, mag

    def tau_con(self):
        return self.log_tau_con.exp().clamp(min=1e-3, max=1.0)


def logits_from_proto(vd, proto, logit_scale):
    vd = F.normalize(vd, dim=-1)
    pr = F.normalize(proto, dim=-1)
    cos = torch.einsum('bd,bkd->bk', vd, pr)
    return logit_scale.exp().clamp(max=100) * cos


def sample_eval_prototypes(fn_list, rng):
    rows = []
    for f in fn_list:
        bank = EVAL_BANK_EMB[f]
        chosen = [bank[c, rng.randrange(bank.size(1))] for c in range(3)]
        rows.append(torch.stack(chosen, 0))
    return torch.stack(rows, 0)


def logits_from_holdout(vd, fn_list, logit_scale, rng):
    proto = sample_eval_prototypes(fn_list, rng).to(vd.device)
    return logits_from_proto(vd, proto, logit_scale)


def _masked_crossmodal_one_way(anchor, other, y_a, y_o, fid_a, fid_o, tau):
    anchor = F.normalize(anchor, dim=-1)
    other = F.normalize(other, dim=-1)
    B, M = anchor.size(0), other.size(0)
    if B == 0 or M == 0:
        return anchor.new_zeros(()), {'n_valid': 0}
    sim = (anchor @ other.t()) / tau
    same_f = fid_a.unsqueeze(1).eq(fid_o.unsqueeze(0))
    same_y = y_a.unsqueeze(1).eq(y_o.unsqueeze(0))
    pos_mask = same_f & same_y
    neg_mask = same_f & ~same_y
    allowed = same_f
    pos_counts = pos_mask.sum(1).float()
    neg_counts = neg_mask.sum(1).float()
    valid = (pos_counts >= 1) & (neg_counts >= 1)
    if not valid.any():
        return anchor.new_zeros(()), {'n_valid': 0}
    neg_large = torch.finfo(sim.dtype).min / 4
    logits = sim.masked_fill(~allowed, neg_large)
    logits = logits - logits.max(1, keepdim=True).values.detach()
    exp_logits = logits.exp() * allowed.float()
    log_prob = logits - exp_logits.sum(1, keepdim=True).clamp(min=1e-8).log()
    pos_log = torch.where(pos_mask, log_prob, torch.zeros_like(log_prob))
    mean_pos = pos_log.sum(1) / pos_counts.clamp(min=1.0)
    loss = -mean_pos[valid].mean()
    if not torch.isfinite(loss):
        loss = anchor.new_zeros(())
    return loss, {'n_valid': int(valid.sum())}


def masked_crossmodal_supcon_loss(d, t, y, fid, tau, symmetric=False):
    t = t.detach()
    li, st = _masked_crossmodal_one_way(d, t, y, y, fid, fid, tau)
    if not symmetric:
        return li, st
    lt, st2 = _masked_crossmodal_one_way(t, d, y, y, fid, fid, tau)
    return 0.5 * (li + lt), {**st, 'n_valid_t2i': st2.get('n_valid', 0)}


def build_buckets(split):
    y, fid = DATA[split]['y'], DATA[split]['fid']
    buckets = defaultdict(list)
    for i in range(len(y)):
        buckets[(int(fid[i]), int(y[i]))].append(i)
    return buckets

TRAIN_BUCKETS = build_buckets('train')

def sample_contrastive_batch(buckets, k_findings=K_FINDINGS_PER_BATCH,
                             n_per_class=N_PER_CLASS, max_bs=MAX_BATCH_SIZE, rng=None):
    rng = rng or random
    by_f = defaultdict(set)
    for (f, y), idxs in buckets.items():
        if idxs:
            by_f[f].add(y)
    eligible = [f for f, ys in by_f.items() if len(ys) >= 2] or list(by_f.keys())
    if not eligible:
        return []
    k = min(k_findings, len(eligible))
    batch = []
    for f in rng.sample(eligible, k):
        for y in range(3):
            pool = buckets.get((f, y), [])
            if not pool:
                continue
            take = min(n_per_class, len(pool))
            batch.extend(rng.sample(pool, take) if len(pool) >= take else list(pool))
    if len(batch) > max_bs:
        batch = rng.sample(batch, max_bs)
    rng.shuffle(batch)
    return batch

_bs = [len(sample_contrastive_batch(TRAIN_BUCKETS)) for _ in range(20)]
APPROX_BS = max(int(sum(_bs) / len(_bs)), 32)
STEPS = max(1, math.ceil(len(DATA['train']['y']) / APPROX_BS))
print('steps', STEPS, 'approx_bs', APPROX_BS)

## 6. Loss ablation under holdout-eval paradigm

Train CE always uses `PROTO_TRAIN`.  
**Tune early-stop and final test** always use **random holdout sentences** (1 of 5 per class).

Sweeps CE × Mag × SupCon (skip all-off). Results → Drive `ctclip_cache/ablations_holdout_eval/`.

In [ ]:
import time
from itertools import product

SKIP_ALL_OFF = True
QUICK_ABLATION = False
ABLATION_EPOCHS = EPOCHS
ABLATION_PATIENCE = PATIENCE
ABLATION_SEED = 0
RUN_TEST_EACH = True
N_EVAL_PASSES = 3

ABL_DIR = f'{DRIVE}/ctclip_cache/ablations_holdout_eval'
os.makedirs(ABL_DIR, exist_ok=True)
ts = time.strftime('%Y%m%d_%H%M%S')
_SYM = globals().get('CONTRASTIVE_SYMMETRIC', False)

def _set_seed(s):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def _make_model():
    return DifferenceTransformer(
        n_findings=len(FINDINGS), d_model=D_MODEL, antisym=ANTISYM,
        magnitude=True, finding_conditioning=FINDING_CONDITIONING,
        finding_as_4th_token=FINDING_AS_4TH_TOKEN,
        use_learned_finding_emb=USE_LEARNED_FINDING_EMB,
        tau_con_init=TAU_CON_INIT, learnable_tau_con=LEARNABLE_TAU_CON,
    ).to(DEVICE)

@torch.no_grad()
def evaluate_holdout(model, split, base_seed, n_passes=N_EVAL_PASSES):
    """Eval with holdout sentences; average macro-F1 over n_passes of resampling."""
    model.eval()
    D = DATA[split]
    macros, pers = [], []
    all_last_y, all_last_p = None, None
    for p in range(n_passes):
        rng = random.Random(base_seed + 10007 * p + 17)
        ys, ps = [], []
        bs = 256
        for i in range(0, len(D['y']), bs):
            sl = slice(i, i + bs)
            vp = D['vp'][sl].to(DEVICE)
            vc = D['vc'][sl].to(DEVICE)
            y = D['y'][sl].to(DEVICE)
            fid = D['fid'][sl].to(DEVICE)
            fn = D['fn'][i:i+bs]
            vd, _ = model(vp, vc, fid)
            lg = logits_from_holdout(vd, fn, model.logit_scale, rng)
            ys += y.cpu().tolist()
            ps += lg.argmax(1).cpu().tolist()
        y_true, y_pred = np.array(ys), np.array(ps)
        macros.append(f1_score(y_true, y_pred, labels=[0,1,2], average='macro', zero_division=0))
        pers.append(f1_score(y_true, y_pred, labels=[0,1,2], average=None, zero_division=0))
        all_last_y, all_last_p = y_true, y_pred
    per = np.mean(np.stack(pers, 0), 0)
    return {
        'macro_f1': float(np.mean(macros)),
        'macro_f1_std': float(np.std(macros)),
        'f1_worsened': float(per[0]),
        'f1_stable': float(per[1]),
        'f1_improved': float(per[2]),
    }

def _train_one(use_ce, use_mag, use_supcon, seed):
    _set_seed(seed)
    model = _make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce_fn = nn.CrossEntropyLoss(weight=W_cls.to(DEVICE))
    best, best_state, best_ep, bad = -1.0, None, 0, 0
    n_train = len(DATA['train']['y'])

    for ep in range(1, ABLATION_EPOCHS + 1):
        model.train()
        tot, n = 0.0, 0
        for _ in range(STEPS):
            if use_supcon:
                idxs = sample_contrastive_batch(TRAIN_BUCKETS)
                if len(idxs) < 4:
                    idxs = random.sample(range(n_train), min(MAX_BATCH_SIZE, n_train))
            else:
                idxs = random.sample(range(n_train), min(MAX_BATCH_SIZE, n_train))
            idxs_t = torch.tensor(idxs, dtype=torch.long)
            vp = DATA['train']['vp'][idxs_t].to(DEVICE)
            vc = DATA['train']['vc'][idxs_t].to(DEVICE)
            pr = DATA['train']['pr'][idxs_t].to(DEVICE)
            y = DATA['train']['y'][idxs_t].to(DEVICE)
            fid = DATA['train']['fid'][idxs_t].to(DEVICE)
            tt = DATA['train']['tt'][idxs_t].to(DEVICE)

            vd, mag = model(vp, vc, fid)
            # TRAIN CE uses fixed PROTO_TRAIN (in pr)
            lg = logits_from_proto(vd, pr, model.logit_scale)

            loss = vd.new_zeros(())
            if use_ce:
                loss = loss + LAMBDA_CE * ce_fn(lg, y)
            if use_mag and mag is not None:
                is_change = (y != C2I['stable']).float()
                loss = loss + LAMBDA_MAG * F.binary_cross_entropy_with_logits(mag, is_change)
            if use_supcon:
                l_con, _ = masked_crossmodal_supcon_loss(
                    vd, tt, y, fid, model.tau_con(), symmetric=_SYM)
                loss = loss + LAMBDA_CON * l_con
            if not torch.isfinite(loss) or (
                loss.item() == 0.0 and not (use_ce or use_mag or use_supcon)
            ):
                loss = vd.sum() * 0.0

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item()) * len(idxs)
            n += len(idxs)

        # early-stop on HOLDOUT eval (not train prototypes)
        tune = evaluate_holdout(model, 'tune', base_seed=seed + ep)
        vf1 = tune['macro_f1']
        if vf1 > best:
            best, best_ep = vf1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if ep % 10 == 0 or ep == 1:
            print('  ep %3d  loss %.3f  tuneHoldoutF1 %.3f+/-%.3f  best %.3f' % (
                ep, tot/max(n,1), vf1, tune['macro_f1_std'], best))
        if bad >= ABLATION_PATIENCE:
            print('  early stop @ ep %d  best %.3f' % (ep, best))
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    return model, best_state, best, best_ep, ep


combos = list(product([0, 1], [0, 1], [0, 1]))
if SKIP_ALL_OFF:
    combos = [c for c in combos if c != (0, 0, 0)]
if QUICK_ABLATION:
    combos = [c for c in combos if c[0] == 1]

print('Holdout-eval ablation | %d combos | epochs<=%d' % (len(combos), ABLATION_EPOCHS))
print('Train CE: PROTO_TRAIN | Eval: random 1-of-%d holdout sents/class x %d passes' % (
    N_EVAL_SENTENCES, N_EVAL_PASSES))
print('combos (CE,Mag,SupCon):', combos)

rows = []
for run_i, (ce, mag, sup) in enumerate(combos):
    tag = 'ce%d_mag%d_supcon%d' % (ce, mag, sup)
    print('')
    print('=' * 60)
    print('[%d/%d] %s' % (run_i + 1, len(combos), tag))
    print('=' * 60)
    seed = ABLATION_SEED + run_i
    t0 = time.time()
    model, state, best_tune, best_ep, epochs_ran = _train_one(
        bool(ce), bool(mag), bool(sup), seed)
    tune_m = evaluate_holdout(model, 'tune', base_seed=seed + 999)
    test_m = evaluate_holdout(model, 'test', base_seed=seed + 1999) if RUN_TEST_EACH else None
    elapsed = time.time() - t0

    row = dict(
        ce=ce, mag=mag, supcon=sup, tag=tag, seed=seed,
        best_epoch=best_ep, epochs_ran=epochs_ran, seconds=round(elapsed, 1),
        tune_macro_f1=tune_m['macro_f1'], tune_macro_f1_std=tune_m['macro_f1_std'],
        tune_f1_worsened=tune_m['f1_worsened'],
        tune_f1_stable=tune_m['f1_stable'],
        tune_f1_improved=tune_m['f1_improved'],
        eval_paradigm='holdout_random_1of5',
    )
    if test_m is not None:
        row.update(
            test_macro_f1=test_m['macro_f1'],
            test_macro_f1_std=test_m['macro_f1_std'],
            test_f1_worsened=test_m['f1_worsened'],
            test_f1_stable=test_m['f1_stable'],
            test_f1_improved=test_m['f1_improved'],
        )
    rows.append(row)

    path = '%s/holdout_ablation_%s_seed%d_%s.pt' % (ABL_DIR, tag, seed, ts)
    torch.save(dict(model=state, row=row, findings=FINDINGS, classes=CLASSES), path)
    print('  saved', path)
    msg = '  tuneHoldoutF1=%.3f+/-%.3f' % (tune_m['macro_f1'], tune_m['macro_f1_std'])
    if test_m is not None:
        msg += '  testHoldoutF1=%.3f+/-%.3f' % (test_m['macro_f1'], test_m['macro_f1_std'])
    msg += '  |  %.1f min' % (elapsed / 60.0)
    print(msg)

df = pd.DataFrame(rows)
sort_col = 'test_macro_f1' if 'test_macro_f1' in df.columns else 'tune_macro_f1'
df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
print('')
print('=' * 60)
print('HOLDOUT-EVAL ABLATION SUMMARY (sorted by %s)' % sort_col)
print('=' * 60)
show = [c for c in [
    'ce', 'mag', 'supcon', 'tune_macro_f1', 'test_macro_f1',
    'test_macro_f1_std', 'test_f1_worsened', 'test_f1_stable', 'test_f1_improved',
    'best_epoch', 'epochs_ran', 'seconds',
] if c in df.columns]
try:
    display(df[show])
except NameError:
    print(df[show].to_string(index=False))

csv_path = '%s/holdout_loss_ablation_%s.csv' % (ABL_DIR, ts)
df.to_csv(csv_path, index=False)
print('wrote', csv_path)
ABLATION_DF = df
print('Done. ABLATION_DF ready. Eval never used train CE prototype strings.')

Holdout-eval ablation | 7 combos | epochs<=120
Train CE: PROTO_TRAIN | Eval: random 1-of-5 holdout sents/class x 3 passes
combos (CE,Mag,SupCon): [(0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]

[1/7] ce0_mag0_supcon1
  ep   1  loss 1.239  tuneHoldoutF1 0.336+/-0.006  best 0.336
  ep  10  loss 1.152  tuneHoldoutF1 0.410+/-0.010  best 0.415
  ep  20  loss 1.124  tuneHoldoutF1 0.397+/-0.011  best 0.434
  ep  30  loss 1.106  tuneHoldoutF1 0.392+/-0.003  best 0.434
  early stop @ ep 37  best 0.434
  saved /content/drive/MyDrive/3dCT/ctclip_cache/ablations_holdout_eval/holdout_ablation_ce0_mag0_supcon1_seed0_20260818_042225.pt
  tuneHoldoutF1=0.435+/-0.011  testHoldoutF1=0.443+/-0.002  |  0.4 min

[2/7] ce0_mag1_supcon0
  ep   1  loss 0.312  tuneHoldoutF1 0.360+/-0.010  best 0.360
  ep  10  loss 0.191  tuneHoldoutF1 0.336+/-0.010  best 0.363
  ep  20  loss 0.133  tuneHoldoutF1 0.335+/-0.016  best 0.363
  early stop @ ep 25  best 0.363
  saved /content/drive/MyD

   ce  mag  supcon  tune_macro_f1  test_macro_f1  test_macro_f1_std  \
0   1    0       0       0.518131       0.538385           0.006105   
1   1    0       1       0.521929       0.518454           0.004199   
2   1    1       0       0.521025       0.514537           0.003933   
3   1    1       1       0.511305       0.482597           0.004602   
4   0    1       1       0.473829       0.464187           0.004137   
5   0    0       1       0.435059       0.442615           0.001639   
6   0    1       0       0.354140       0.343902           0.009904   

   test_f1_worsened  test_f1_stable  test_f1_improved  best_epoch  epochs_ran  \
0          0.529194        0.412734          0.673227           2          22   
1          0.534622        0.445413          0.575326          77          97   
2          0.447027        0.451931          0.644653           2          22   
3          0.518557        0.388173          0.541060          46          66   
4          0.432571       

wrote /content/drive/MyDrive/3dCT/ctclip_cache/ablations_holdout_eval/holdout_loss_ablation_20260818_042225.csv
Done. ABLATION_DF ready. Eval never used train CE prototype strings.
